# BrewScope — Stage 5: Text Preprocessing

Notebook ini menyiapkan hasil labeling ABSA menjadi dataset modeling yang bersih, dapat diaudit, dan siap digunakan oleh SVM, Random Forest, dan Multinomial Naive Bayes.

Prinsip utama:

- teks asli tidak pernah ditimpa;
- negasi, emoji, dan penanda intensitas dipertahankan karena penting untuk sentimen;
- pembersihan bersifat deterministik dan tidak mempelajari informasi dari keseluruhan korpus;
- duplikat ditandai sebagai kelompok untuk mencegah leakage, bukan langsung dihapus;
- tersedia beberapa varian teks agar dampak stemming dan stopword removal dapat dibandingkan;
- vectorizer belum dibuat di notebook ini dan nantinya wajib di-fit hanya pada training set;
- label otomatis telah disetujui tim/dosen untuk dilanjutkan, tetapi provenance label tetap dicatat.

Output utama:

1. data/processed/modeling/reviews_preprocessed.csv
2. data/processed/modeling/aspect_detection_dataset.csv
3. data/processed/modeling/aspect_sentiment_dataset.csv
4. data/processed/modeling/preprocessing_manifest.json
5. visual paper-ready di reports/figures/preprocessing/
6. tabel ringkasan di reports/tables/preprocessing/

## 0. Persiapan environment

Install dependency preprocessing dari root repository sebelum menjalankan notebook:

    python -m pip install -e ".[preprocessing]"

Notebook sengaja tidak mengunduh kamus atau resource dari internet saat runtime. Dengan demikian, hasil preprocessing tidak berubah karena sumber eksternal.

In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import re
import unicodedata
from collections import Counter
from functools import lru_cache
from pathlib import Path

import emoji
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "savefig.dpi": 300,
        "axes.titleweight": "bold",
        "axes.labelweight": "semibold",
        "font.family": "DejaVu Sans",
    }
)

RANDOM_STATE = 42
ALLOW_DRAFT_LABELS = True  # Keputusan tim/dosen: label boleh masuk preprocessing.

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Root proyek tidak ditemukan. Jalankan notebook dari root repo atau folder notebooks/."
    )

PAIR_INPUT = (
    PROJECT_ROOT / "data" / "processed" / "labeling" / "annotations_flat.csv"
)
REVIEW_SUMMARY_INPUT = (
    PROJECT_ROOT / "data" / "processed" / "labeling" / "review_level_summary.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "modeling"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures" / "preprocessing"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "preprocessing"

ASPECTS = [
    "product_quality",
    "service",
    "price_value",
    "ambience",
    "facilities",
    "cleanliness",
]
SENTIMENTS = ["negative", "neutral", "positive"]

print(f"Project root : {PROJECT_ROOT}")
print(f"Input pairs  : {PAIR_INPUT}")
print(f"Output dir   : {OUTPUT_DIR}")

## 1. Memuat dan memvalidasi hasil labeling

Input utama adalah annotations_flat.csv. Satu review seharusnya memiliki tepat enam baris—masing-masing satu keputusan untuk setiap aspek.

In [ ]:
REQUIRED_PAIR_COLUMNS = {
    "annotation_id",
    "review_id",
    "coffee_shop_id",
    "review_text",
    "aspect_category",
    "aspect_presence",
    "sentiment",
    "mixed_sentiment",
    "annotation_status",
    "training_eligible",
    "review_flags_json",
}

pairs = pd.read_csv(
    PAIR_INPUT,
    encoding="utf-8-sig",
    keep_default_na=False,
    dtype={
        "annotation_id": "string",
        "review_id": "string",
        "coffee_shop_id": "string",
        "review_text": "string",
        "aspect_category": "string",
        "aspect_presence": "string",
        "sentiment": "string",
        "annotation_status": "string",
    },
)
review_summary = pd.read_csv(
    REVIEW_SUMMARY_INPUT,
    encoding="utf-8-sig",
    keep_default_na=False,
    dtype={
        "review_id": "string",
        "coffee_shop_id": "string",
        "review_text": "string",
    },
)

missing_columns = sorted(REQUIRED_PAIR_COLUMNS - set(pairs.columns))
if missing_columns:
    raise ValueError(f"Kolom input belum lengkap: {missing_columns}")

def parse_boolean(series: pd.Series, column_name: str) -> pd.Series:
    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }
    normalized = series.astype("string").str.strip().str.casefold()
    parsed = normalized.map(mapping)
    if parsed.isna().any():
        examples = normalized[parsed.isna()].drop_duplicates().head(5).tolist()
        raise ValueError(f"Nilai boolean tidak valid pada {column_name}: {examples}")
    return parsed.astype(bool)

pairs["mixed_sentiment"] = parse_boolean(
    pairs["mixed_sentiment"], "mixed_sentiment"
)
pairs["training_eligible"] = parse_boolean(
    pairs["training_eligible"], "training_eligible"
)

if pairs["annotation_id"].duplicated().any():
    raise ValueError("annotation_id duplikat ditemukan.")

if pairs.duplicated(["review_id", "aspect_category"]).any():
    raise ValueError("Pasangan review_id dan aspect_category duplikat ditemukan.")

pair_completeness = pairs.groupby("review_id").agg(
    pair_rows=("aspect_category", "size"),
    unique_aspects=("aspect_category", "nunique"),
)
if not (
    pair_completeness["pair_rows"].eq(len(ASPECTS))
    & pair_completeness["unique_aspects"].eq(len(ASPECTS))
).all():
    raise ValueError("Tidak semua review memiliki tepat enam keputusan aspek.")

observed_aspects = set(pairs["aspect_category"].unique())
if observed_aspects != set(ASPECTS):
    raise ValueError(
        f"Set aspek tidak sesuai. Ditemukan: {sorted(observed_aspects)}"
    )

input_overview = pd.DataFrame(
    {
        "metric": [
            "Pasangan review-aspek",
            "Review unik",
            "Coffee shop",
            "Annotation ID unik",
            "Label uncertain",
            "Mixed sentiment",
            "Training eligible dari sumber",
        ],
        "value": [
            len(pairs),
            pairs["review_id"].nunique(),
            pairs["coffee_shop_id"].nunique(),
            pairs["annotation_id"].nunique(),
            pairs["annotation_status"].eq("uncertain").sum(),
            pairs["mixed_sentiment"].sum(),
            pairs["training_eligible"].sum(),
        ],
    }
)
display(input_overview)
display(pairs.head(3))

## 2. Keputusan metodologis

Teknik dari project sebelumnya yang dipertahankan:

- case folding;
- pembersihan URL dan HTML;
- normalisasi kata tidak baku;
- tokenisasi;
- stopword removal;
- stemming Bahasa Indonesia;
- perbandingan teks sebelum dan sesudah preprocessing;
- visual frekuensi kata.

Perbaikan untuk ABSA:

- emoji diterjemahkan menjadi token, bukan dibuang;
- negasi seperti tidak, bukan, belum, dan kurang tidak ikut dibuang;
- angka, URL, email, dan telepon diganti placeholder agar informasi keberadaannya tetap ada;
- kamus slang disimpan lokal agar reproducible;
- teks asli, teks bersih, teks tanpa stopword, teks dengan negasi, dan teks stemming disimpan terpisah;
- duplikat ditandai untuk group-aware split;
- label aspek dan sentimen dipisahkan menjadi dua dataset modeling;
- visual menggunakan frekuensi dan persentase yang dapat dipertanggungjawabkan, bukan hanya word cloud.

## 3. Resource normalisasi Bahasa Indonesia

Kamus berikut fokus pada slang yang umum muncul dalam ulasan. Kamus dapat diperluas secara versioned setelah audit token, tetapi jangan mengubah makna opini.

In [ ]:
SLANG_MAP = {
    "gak": "tidak",
    "ga": "tidak",
    "gk": "tidak",
    "nggak": "tidak",
    "ngga": "tidak",
    "enggak": "tidak",
    "kagak": "tidak",
    "tak": "tidak",
    "blm": "belum",
    "belom": "belum",
    "bgt": "banget",
    "bgtt": "banget",
    "bgttt": "banget",
    "bangett": "banget",
    "bnget": "banget",
    "bngt": "banget",
    "yg": "yang",
    "dgn": "dengan",
    "dg": "dengan",
    "sm": "sama",
    "sma": "sama",
    "utk": "untuk",
    "buat": "untuk",
    "dr": "dari",
    "krn": "karena",
    "karna": "karena",
    "klo": "kalau",
    "kl": "kalau",
    "kalo": "kalau",
    "tp": "tapi",
    "tpi": "tapi",
    "udh": "sudah",
    "udah": "sudah",
    "sdh": "sudah",
    "jd": "jadi",
    "jdi": "jadi",
    "emg": "memang",
    "emang": "memang",
    "bkin": "bikin",
    "bwt": "untuk",
    "aja": "saja",
    "doang": "saja",
    "cmn": "cuma",
    "cuman": "cuma",
    "trs": "terus",
    "trus": "terus",
    "lg": "lagi",
    "lgi": "lagi",
    "pdhl": "padahal",
    "kyk": "seperti",
    "kayak": "seperti",
    "gmn": "bagaimana",
    "gimana": "bagaimana",
    "knp": "kenapa",
    "dmn": "dimana",
    "skrg": "sekarang",
    "skrng": "sekarang",
    "bbrp": "beberapa",
    "org": "orang",
    "org2": "orang",
    "gw": "saya",
    "gue": "saya",
    "aku": "saya",
    "sy": "saya",
    "lu": "kamu",
    "lo": "kamu",
    "km": "kamu",
    "tmpt": "tempat",
    "tempatnyaa": "tempatnya",
    "rmah": "ramah",
    "mantep": "mantap",
    "mantul": "mantap",
    "rekomen": "rekomendasi",
    "recommended": "rekomendasi",
    "coffe": "coffee",
    "cappucino": "cappuccino",
    "servis": "service",
    "antri": "antre",
    "pesen": "pesan",
    "pesenan": "pesanan",
    "dateng": "datang",
    "mkn": "makan",
    "mnm": "minum",
    "hrg": "harga",
    "murmer": "murah",
    "nyamann": "nyaman",
    "enaak": "enak",
}

EMOTICON_MAP = {
    ":-)": " emoticon_positive ",
    ":)": " emoticon_positive ",
    ":d": " emoticon_positive ",
    "xd": " emoticon_positive ",
    ":-d": " emoticon_positive ",
    ":(": " emoticon_negative ",
    ":-(": " emoticon_negative ",
    ":'(": " emoticon_negative ",
    ":/": " emoticon_negative ",
}

NEGATIONS = {"tidak", "bukan", "belum", "jangan", "kurang", "tanpa"}
INTENSIFIERS = {
    "sangat",
    "banget",
    "amat",
    "terlalu",
    "lebih",
    "paling",
    "sekali",
}
PROTECTED_STOPWORDS = NEGATIONS | INTENSIFIERS | {
    "tapi",
    "tetapi",
    "namun",
    "meskipun",
    "walaupun",
}

sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
CUSTOM_STOPWORDS = {
    "nih",
    "sih",
    "deh",
    "dong",
    "lah",
    "ya",
    "yah",
    "kok",
}
STOPWORDS = (sastrawi_stopwords | CUSTOM_STOPWORDS) - PROTECTED_STOPWORDS

stemmer = StemmerFactory().create_stemmer()

print(f"Slang dictionary : {len(SLANG_MAP):,} entries")
print(f"Stopwords aktif  : {len(STOPWORDS):,} kata")
print(f"Stopwords dijaga : {sorted(PROTECTED_STOPWORDS)}")

## 4. Fungsi preprocessing

Urutan transformasi:

1. Unicode normalization dan HTML unescape.
2. Placeholder untuk URL, email, nomor telepon, dan angka.
3. Emoji dan emoticon menjadi token semantik.
4. Case folding.
5. Normalisasi karakter berulang.
6. Tokenisasi.
7. Normalisasi slang.
8. Pembuatan varian negasi, stopword, dan stemming.

text_clean digunakan sebagai baseline utama karena paling sedikit kehilangan informasi. Varian lain dibandingkan saat modeling.

In [ ]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
EMAIL_PATTERN = re.compile(r"\b[\w.+-]+@[\w.-]+\.[a-z]{2,}\b", re.IGNORECASE)
HTML_PATTERN = re.compile(r"<[^>]+>")
PHONE_PATTERN = re.compile(
    r"(?<!\w)(?:\+?62|0)[\s.-]?(?:\d[\s.-]?){8,13}(?!\w)"
)
NUMBER_PATTERN = re.compile(r"\b\d+(?:[.,]\d+)?\b")
MENTION_PATTERN = re.compile(r"(?<!\w)@[a-z0-9_]+", re.IGNORECASE)
ZERO_WIDTH_PATTERN = re.compile(r"[\u200b-\u200f\u2060\ufeff]")
NON_TOKEN_PATTERN = re.compile(r"[^a-z0-9_\s]")
TOKEN_PATTERN = re.compile(r"[a-z]+(?:_[a-z]+)*|[0-9]+")
WHITESPACE_PATTERN = re.compile(r"\s+")
REPEATED_CHARACTER_PATTERN = re.compile(r"([a-z])\1{2,}")


def replace_emoticons(text: str) -> str:
    result = text
    for emoticon, replacement in sorted(
        EMOTICON_MAP.items(), key=lambda item: len(item[0]), reverse=True
    ):
        result = result.replace(emoticon, replacement)
    return result


def normalize_unicode(text: str) -> str:
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    return ZERO_WIDTH_PATTERN.sub(" ", text)


def normalize_repeated_characters(text: str) -> str:
    # Tiga karakter atau lebih dipendekkan menjadi dua: enaaak -> enaak.
    # Dua karakter dipertahankan agar kata alami seperti coffee tidak rusak.
    return REPEATED_CHARACTER_PATTERN.sub(r"\1\1", text)


def tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text)


def normalize_slang(tokens: list[str]) -> list[str]:
    return [SLANG_MAP.get(token, token) for token in tokens]


def bind_negation(tokens: list[str]) -> list[str]:
    bound: list[str] = []
    index = 0
    while index < len(tokens):
        token = tokens[index]
        if token in NEGATIONS and index + 1 < len(tokens):
            bound.append(f"{token}_{tokens[index + 1]}")
            index += 2
        else:
            bound.append(token)
            index += 1
    return bound


def remove_stopwords(tokens: list[str]) -> list[str]:
    return [token for token in tokens if token not in STOPWORDS]


@lru_cache(maxsize=50_000)
def stem_token(token: str) -> str:
    # Placeholder, emoji, emoticon, dan negation-bound token tidak di-stem.
    if "_" in token or token.endswith("token"):
        return token
    return stemmer.stem(token)


def stem_tokens(tokens: list[str]) -> list[str]:
    return [stem_token(token) for token in tokens]


def preprocess_text(value: object) -> dict[str, object]:
    raw = "" if pd.isna(value) else str(value)
    normalized = normalize_unicode(raw)

    audit = {
        "had_url": bool(URL_PATTERN.search(normalized)),
        "had_email": bool(EMAIL_PATTERN.search(normalized)),
        "had_phone": bool(PHONE_PATTERN.search(normalized)),
        "had_number": bool(NUMBER_PATTERN.search(normalized)),
        "had_html": bool(HTML_PATTERN.search(normalized)),
        "had_emoji": emoji.emoji_count(normalized) > 0,
        "had_repeated_characters": bool(
            REPEATED_CHARACTER_PATTERN.search(normalized.casefold())
        ),
    }

    text = HTML_PATTERN.sub(" ", normalized)
    text = URL_PATTERN.sub(" urltoken ", text)
    text = EMAIL_PATTERN.sub(" emailtoken ", text)
    text = PHONE_PATTERN.sub(" phonetoken ", text)
    text = MENTION_PATTERN.sub(" usertoken ", text)
    text = NUMBER_PATTERN.sub(" numtoken ", text)
    text = replace_emoticons(text.casefold())
    text = emoji.demojize(text, delimiters=(" emoji_", " "))
    text = normalize_repeated_characters(text)
    text = NON_TOKEN_PATTERN.sub(" ", text)
    text = WHITESPACE_PATTERN.sub(" ", text).strip()

    tokens = normalize_slang(tokenize(text))
    clean_tokens = [token for token in tokens if token]
    no_stopword_tokens = remove_stopwords(clean_tokens)
    negation_tokens = bind_negation(clean_tokens)
    stemmed_tokens = stem_tokens(no_stopword_tokens)

    result = {
        "text_clean": " ".join(clean_tokens),
        "text_negation": " ".join(negation_tokens),
        "text_no_stopwords": " ".join(no_stopword_tokens),
        "text_stemmed": " ".join(stemmed_tokens),
        "text_model": " ".join(clean_tokens),
        "tokens_clean": clean_tokens,
        "tokens_no_stopwords": no_stopword_tokens,
        "raw_char_count": len(raw),
        "clean_char_count": len(" ".join(clean_tokens)),
        "raw_token_count": len(re.findall(r"\b\w+\b", raw, flags=re.UNICODE)),
        "clean_token_count": len(clean_tokens),
    }
    return {**result, **audit}

## 5. Menjalankan preprocessing satu kali per review

Preprocessing dilakukan pada 3.538 review unik, bukan diulang enam kali untuk setiap aspek.

In [ ]:
review_text_consistency = pairs.groupby("review_id").agg(
    text_versions=("review_text", "nunique"),
    shop_versions=("coffee_shop_id", "nunique"),
)
if not (
    review_text_consistency["text_versions"].eq(1)
    & review_text_consistency["shop_versions"].eq(1)
).all():
    raise ValueError("Satu review_id memiliki teks atau coffee_shop_id yang tidak konsisten.")

reviews = (
    pairs[["review_id", "coffee_shop_id", "review_text"]]
    .drop_duplicates("review_id")
    .rename(columns={"review_text": "review_text_raw"})
    .reset_index(drop=True)
)

metadata_columns = [
    "review_id",
    "source_row_number",
    "review_flags",
    "review_needs_manual_review",
    "needs_language_review_in_source",
    "needs_privacy_review_in_source",
]
available_metadata = [
    column for column in metadata_columns if column in review_summary.columns
]
reviews = reviews.merge(
    review_summary[available_metadata].drop_duplicates("review_id"),
    on="review_id",
    how="left",
    validate="one_to_one",
)

processed_columns = reviews["review_text_raw"].map(preprocess_text).apply(pd.Series)
reviews = pd.concat([reviews, processed_columns], axis=1)

reviews["text_changed"] = (
    reviews["review_text_raw"].str.casefold().str.strip() != reviews["text_clean"]
)
reviews["clean_text_empty"] = reviews["text_clean"].str.strip().eq("")

display(
    reviews[
        [
            "review_id",
            "review_text_raw",
            "text_clean",
            "text_negation",
            "text_no_stopwords",
            "text_stemmed",
        ]
    ].head(10)
)

## 6. Audit kualitas teks dan duplicate grouping

Review tidak langsung dibuang hanya karena teksnya sama. Setiap review diberi duplicate_group_id berdasarkan text_clean. Kolom ini wajib digunakan saat train/validation/test split agar teks identik tidak masuk ke split berbeda.

In [ ]:
def stable_group_id(text: str, prefix: str = "TXT") -> str:
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16].upper()
    return f"{prefix}_{digest}"


reviews["duplicate_group_id"] = reviews["text_clean"].map(stable_group_id)
reviews["duplicate_group_size"] = reviews.groupby("duplicate_group_id")[
    "review_id"
].transform("size")
reviews["is_normalized_duplicate"] = reviews["duplicate_group_size"].gt(1)

quality_summary = pd.DataFrame(
    {
        "metric": [
            "Review unik",
            "Teks berubah setelah preprocessing",
            "Teks kosong setelah preprocessing",
            "Review pada duplicate group",
            "Duplicate group unik",
            "Mengandung emoji",
            "Mengandung URL",
            "Mengandung nomor",
            "Mengandung karakter berulang",
            "Median token sebelum",
            "Median token sesudah",
        ],
        "value": [
            len(reviews),
            reviews["text_changed"].sum(),
            reviews["clean_text_empty"].sum(),
            reviews["is_normalized_duplicate"].sum(),
            reviews["duplicate_group_id"].nunique(),
            reviews["had_emoji"].sum(),
            reviews["had_url"].sum(),
            reviews["had_number"].sum(),
            reviews["had_repeated_characters"].sum(),
            reviews["raw_token_count"].median(),
            reviews["clean_token_count"].median(),
        ],
    }
)

display(quality_summary)
display(
    reviews.loc[
        reviews["is_normalized_duplicate"],
        [
            "duplicate_group_id",
            "duplicate_group_size",
            "review_id",
            "review_text_raw",
            "text_clean",
        ],
    ]
    .sort_values(["duplicate_group_size", "duplicate_group_id"], ascending=[False, True])
    .head(20)
)

if reviews["clean_text_empty"].any():
    display(
        reviews.loc[
            reviews["clean_text_empty"],
            ["review_id", "review_text_raw", "text_clean"],
        ]
    )

## 7. Helper visualisasi paper-ready

Semua gambar disimpan sebagai PNG 300 DPI. Angka tetap ditampilkan agar grafik dapat diperiksa dan ditulis di bagian hasil paper.

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(filename: str) -> None:
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {path.relative_to(PROJECT_ROOT)}")


def add_bar_labels(axis, fmt: str = "{:.0f}") -> None:
    for container in axis.containers:
        axis.bar_label(container, fmt=fmt, padding=3, fontsize=8)


def top_tokens(series: pd.Series, n: int = 20) -> pd.DataFrame:
    counter = Counter()
    for text in series.dropna().astype(str):
        counter.update(token for token in text.split() if token)
    return pd.DataFrame(counter.most_common(n), columns=["token", "frequency"])

### 7.1 Distribusi panjang teks sebelum dan sesudah preprocessing

In [ ]:
length_plot = reviews[
    ["raw_token_count", "clean_token_count"]
].rename(
    columns={
        "raw_token_count": "Sebelum preprocessing",
        "clean_token_count": "Sesudah preprocessing",
    }
).melt(var_name="Tahap", value_name="Jumlah token")

plt.figure(figsize=(11, 5.5))
sns.histplot(
    data=length_plot,
    x="Jumlah token",
    hue="Tahap",
    bins=40,
    element="step",
    stat="density",
    common_norm=False,
    alpha=0.25,
)
plt.title("Distribusi Panjang Review Sebelum dan Sesudah Preprocessing")
plt.xlabel("Jumlah token per review")
plt.ylabel("Density")
save_figure("01_token_length_before_after.png")

### 7.2 Token paling sering sebelum dan sesudah preprocessing

Grafik frekuensi lebih informatif daripada word cloud karena jumlah kemunculan dapat dibaca secara langsung.

In [ ]:
raw_for_frequency = (
    reviews["review_text_raw"]
    .astype(str)
    .str.casefold()
    .str.replace(r"[^a-z0-9\s]", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
top_raw = top_tokens(raw_for_frequency, n=20)
top_clean = top_tokens(reviews["text_clean"], n=20)

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
sns.barplot(
    data=top_raw,
    x="frequency",
    y="token",
    ax=axes[0],
    color="#7AA6C2",
)
axes[0].set_title("20 Token Teratas — Teks Mentah")
axes[0].set_xlabel("Frekuensi")
axes[0].set_ylabel("Token")
add_bar_labels(axes[0])

sns.barplot(
    data=top_clean,
    x="frequency",
    y="token",
    ax=axes[1],
    color="#D98E73",
)
axes[1].set_title("20 Token Teratas — Teks Bersih")
axes[1].set_xlabel("Frekuensi")
axes[1].set_ylabel("Token")
add_bar_labels(axes[1])

save_figure("02_top_tokens_before_after.png")

### 7.3 Distribusi mention aspek

In [ ]:
aspect_distribution = (
    pairs.assign(is_mentioned=pairs["aspect_presence"].eq("mentioned"))
    .groupby("aspect_category", as_index=False)
    .agg(
        mentioned=("is_mentioned", "sum"),
        total=("review_id", "size"),
    )
)
aspect_distribution["percentage"] = (
    100 * aspect_distribution["mentioned"] / aspect_distribution["total"]
)
aspect_distribution["aspect_category"] = pd.Categorical(
    aspect_distribution["aspect_category"],
    categories=ASPECTS,
    ordered=True,
)
aspect_distribution = aspect_distribution.sort_values("aspect_category")

plt.figure(figsize=(10, 5.5))
axis = sns.barplot(
    data=aspect_distribution,
    x="aspect_category",
    y="mentioned",
    color="#4C78A8",
)
add_bar_labels(axis)
plt.title("Jumlah Review yang Menyebut Setiap Aspek")
plt.xlabel("Aspek")
plt.ylabel("Jumlah mention")
plt.xticks(rotation=25, ha="right")
save_figure("03_aspect_mention_distribution.png")

display(aspect_distribution)

### 7.4 Distribusi sentimen per aspek

Hanya pasangan mentioned dengan sentimen tiga kelas yang terselesaikan dan bukan mixed sentiment yang masuk grafik.

In [ ]:
resolved_sentiment = pairs.loc[
    pairs["aspect_presence"].eq("mentioned")
    & pairs["sentiment"].isin(SENTIMENTS)
    & ~pairs["mixed_sentiment"]
    & pairs["annotation_status"].ne("uncertain"),
    ["review_id", "aspect_category", "sentiment"],
].copy()

sentiment_counts = (
    resolved_sentiment.groupby(["aspect_category", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=ASPECTS, columns=SENTIMENTS, fill_value=0)
)
sentiment_percentages = sentiment_counts.div(
    sentiment_counts.sum(axis=1).replace(0, np.nan),
    axis=0,
).mul(100)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(
    sentiment_counts,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    ax=axes[0],
)
axes[0].set_title("Jumlah Label Sentimen per Aspek")
axes[0].set_xlabel("Sentimen")
axes[0].set_ylabel("Aspek")

sns.heatmap(
    sentiment_percentages,
    annot=True,
    fmt=".1f",
    cmap="YlOrBr",
    cbar_kws={"label": "Persentase (%)"},
    ax=axes[1],
)
axes[1].set_title("Persentase Sentimen dalam Setiap Aspek")
axes[1].set_xlabel("Sentimen")
axes[1].set_ylabel("Aspek")

save_figure("04_sentiment_distribution_by_aspect.png")

### 7.5 Dampak operasi pembersihan

In [ ]:
impact_columns = {
    "had_emoji": "Emoji",
    "had_url": "URL",
    "had_email": "Email",
    "had_phone": "Nomor telepon",
    "had_number": "Angka",
    "had_html": "HTML",
    "had_repeated_characters": "Karakter berulang",
    "text_changed": "Teks berubah",
    "is_normalized_duplicate": "Duplikat ternormalisasi",
    "clean_text_empty": "Teks bersih kosong",
}
preprocessing_impact = pd.DataFrame(
    {
        "operation": list(impact_columns.values()),
        "reviews": [
            int(reviews[column].sum()) for column in impact_columns
        ],
    }
).sort_values("reviews", ascending=False)

plt.figure(figsize=(10, 5.5))
axis = sns.barplot(
    data=preprocessing_impact,
    x="reviews",
    y="operation",
    color="#59A14F",
)
add_bar_labels(axis)
plt.title("Dampak Preprocessing pada Korpus Review")
plt.xlabel("Jumlah review")
plt.ylabel("Kondisi")
save_figure("05_preprocessing_impact.png")

display(preprocessing_impact)

## 8. Membentuk dataset Task A — Aspect Detection

Task A adalah klasifikasi multi-label pada tingkat review. Review dengan presence uncertain pada salah satu aspek tidak dimasukkan karena target enam aspek harus lengkap dan tegas.

In [ ]:
presence_matrix = pairs.pivot(
    index="review_id",
    columns="aspect_category",
    values="aspect_presence",
).reindex(columns=ASPECTS)

task_a_mask = presence_matrix.isin(["mentioned", "not_mentioned"]).all(axis=1)
if not ALLOW_DRAFT_LABELS:
    eligible_by_review = pairs.groupby("review_id")["training_eligible"].all()
    task_a_mask &= eligible_by_review.reindex(presence_matrix.index).fillna(False)

aspect_targets = (
    presence_matrix.loc[task_a_mask]
    .eq("mentioned")
    .astype("int8")
    .rename(columns={aspect: f"aspect_{aspect}" for aspect in ASPECTS})
    .reset_index()
)

MODEL_TEXT_COLUMNS = [
    "review_id",
    "coffee_shop_id",
    "review_text_raw",
    "text_clean",
    "text_negation",
    "text_no_stopwords",
    "text_stemmed",
    "text_model",
    "duplicate_group_id",
    "duplicate_group_size",
]
task_a = reviews[MODEL_TEXT_COLUMNS].merge(
    aspect_targets,
    on="review_id",
    how="inner",
    validate="one_to_one",
)
task_a["label_policy"] = (
    "draft_labels_approved" if ALLOW_DRAFT_LABELS else "training_eligible_only"
)

ASPECT_LABEL_COLUMNS = [f"aspect_{aspect}" for aspect in ASPECTS]
if not task_a[ASPECT_LABEL_COLUMNS].isin([0, 1]).all().all():
    raise ValueError("Target Task A harus biner.")

print(f"Task A rows: {len(task_a):,}")
display(task_a.head(3))
display(task_a[ASPECT_LABEL_COLUMNS].sum().rename("positive_labels").to_frame())

## 9. Membentuk dataset Task B — Aspect Sentiment Classification

Task B menggunakan pasangan review-aspek yang:

- aspek presence-nya mentioned;
- memiliki label positive, neutral, atau negative;
- bukan mixed sentiment;
- bukan annotation_status uncertain.

Satu review dapat muncul beberapa kali karena membahas beberapa aspek. duplicate_group_id tetap dipertahankan agar seluruh pasangan dari review/teks yang sama masuk ke split yang sama.

In [ ]:
task_b_mask = (
    pairs["aspect_presence"].eq("mentioned")
    & pairs["sentiment"].isin(SENTIMENTS)
    & ~pairs["mixed_sentiment"]
    & pairs["annotation_status"].ne("uncertain")
)
if not ALLOW_DRAFT_LABELS:
    task_b_mask &= pairs["training_eligible"]

task_b_pairs = pairs.loc[
    task_b_mask,
    [
        "annotation_id",
        "review_id",
        "aspect_category",
        "sentiment",
        "mention_type",
        "annotation_status",
        "annotator_id",
        "taxonomy_version",
    ],
].copy()

task_b = task_b_pairs.merge(
    reviews[MODEL_TEXT_COLUMNS],
    on="review_id",
    how="inner",
    validate="many_to_one",
)
task_b["label_policy"] = (
    "draft_labels_approved" if ALLOW_DRAFT_LABELS else "training_eligible_only"
)

if not set(task_b["sentiment"].unique()).issubset(SENTIMENTS):
    raise ValueError("Task B mengandung kelas sentimen yang tidak dikenal.")

print(f"Task B rows: {len(task_b):,}")
display(task_b.head(3))
display(
    task_b.groupby(["aspect_category", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=ASPECTS, columns=SENTIMENTS, fill_value=0)
)

## 10. Ringkasan dataset untuk paper

In [ ]:
paper_summary = pd.DataFrame(
    {
        "metric": [
            "Review input",
            "Review Task A",
            "Review dikeluarkan dari Task A karena uncertain",
            "Pasangan review-aspek input",
            "Pasangan Task B",
            "Mixed sentiment dikeluarkan",
            "Uncertain pair",
            "Coffee shop",
            "Duplicate group",
            "Teks bersih kosong",
        ],
        "value": [
            pairs["review_id"].nunique(),
            task_a["review_id"].nunique(),
            pairs["review_id"].nunique() - task_a["review_id"].nunique(),
            len(pairs),
            len(task_b),
            pairs["mixed_sentiment"].sum(),
            pairs["annotation_status"].eq("uncertain").sum(),
            pairs["coffee_shop_id"].nunique(),
            reviews["duplicate_group_id"].nunique(),
            reviews["clean_text_empty"].sum(),
        ],
    }
)
display(paper_summary)

paper_summary.to_csv(
    TABLE_DIR / "preprocessing_dataset_summary.csv",
    index=False,
    encoding="utf-8-sig",
)
aspect_distribution.to_csv(
    TABLE_DIR / "aspect_mention_distribution.csv",
    index=False,
    encoding="utf-8-sig",
)
sentiment_counts.to_csv(
    TABLE_DIR / "sentiment_counts_by_aspect.csv",
    encoding="utf-8-sig",
)
sentiment_percentages.to_csv(
    TABLE_DIR / "sentiment_percentages_by_aspect.csv",
    encoding="utf-8-sig",
)

## 11. Validasi akhir sebelum ekspor

Cell ini memastikan teks asli tetap tersedia, output tidak kosong, label sesuai kontrak, dan duplicate grouping tersedia untuk tahap splitting.

In [ ]:
if reviews["review_id"].duplicated().any():
    raise ValueError("reviews_preprocessed masih memiliki review_id duplikat.")

if reviews["review_text_raw"].isna().any():
    raise ValueError("Teks asli tidak boleh hilang.")

if reviews["clean_text_empty"].any():
    raise ValueError(
        "Ada teks kosong setelah preprocessing. Audit sebelum melanjutkan ekspor."
    )

if task_a.empty or task_b.empty:
    raise ValueError("Dataset modeling tidak boleh kosong.")

if task_a["duplicate_group_id"].isna().any():
    raise ValueError("Task A memiliki duplicate_group_id kosong.")

if task_b["duplicate_group_id"].isna().any():
    raise ValueError("Task B memiliki duplicate_group_id kosong.")

if not task_b["sentiment"].isin(SENTIMENTS).all():
    raise ValueError("Label Task B di luar tiga kelas.")

if not task_a[ASPECT_LABEL_COLUMNS].isin([0, 1]).all().all():
    raise ValueError("Label Task A tidak biner.")

final_checks = pd.DataFrame(
    {
        "check": [
            "Review ID unik",
            "Teks mentah tersedia",
            "Teks model tidak kosong",
            "Task A tidak kosong",
            "Task B tidak kosong",
            "Task A label biner",
            "Task B label tiga kelas",
            "Duplicate group tersedia",
        ],
        "passed": [
            reviews["review_id"].is_unique,
            reviews["review_text_raw"].notna().all(),
            reviews["text_model"].str.strip().ne("").all(),
            not task_a.empty,
            not task_b.empty,
            task_a[ASPECT_LABEL_COLUMNS].isin([0, 1]).all().all(),
            task_b["sentiment"].isin(SENTIMENTS).all(),
            task_a["duplicate_group_id"].notna().all()
            and task_b["duplicate_group_id"].notna().all(),
        ],
    }
)
display(final_checks)

## 12. Ekspor artefak preprocessing

Dataset diekspor setelah seluruh pemeriksaan lulus. File manifest mencatat hash input/output, konfigurasi, jumlah data, dan kebijakan label.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

review_output = OUTPUT_DIR / "reviews_preprocessed.csv"
task_a_output = OUTPUT_DIR / "aspect_detection_dataset.csv"
task_b_output = OUTPUT_DIR / "aspect_sentiment_dataset.csv"
manifest_output = OUTPUT_DIR / "preprocessing_manifest.json"

reviews_export = reviews.drop(
    columns=["tokens_clean", "tokens_no_stopwords"],
    errors="ignore",
)
reviews_export.to_csv(review_output, index=False, encoding="utf-8-sig")
task_a.to_csv(task_a_output, index=False, encoding="utf-8-sig")
task_b.to_csv(task_b_output, index=False, encoding="utf-8-sig")


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


manifest = {
    "stage": "text_preprocessing",
    "input": {
        "annotations_flat": str(PAIR_INPUT.relative_to(PROJECT_ROOT)),
        "annotations_flat_sha256": sha256_file(PAIR_INPUT),
        "review_level_summary": str(REVIEW_SUMMARY_INPUT.relative_to(PROJECT_ROOT)),
        "review_level_summary_sha256": sha256_file(REVIEW_SUMMARY_INPUT),
    },
    "configuration": {
        "allow_draft_labels": ALLOW_DRAFT_LABELS,
        "primary_text_column": "text_model",
        "primary_text_variant": "text_clean",
        "random_state_for_next_stage": RANDOM_STATE,
        "aspects": ASPECTS,
        "sentiments": SENTIMENTS,
        "slang_dictionary_entries": len(SLANG_MAP),
        "stopword_count": len(STOPWORDS),
        "protected_stopwords": sorted(PROTECTED_STOPWORDS),
    },
    "counts": {
        "reviews_preprocessed": len(reviews_export),
        "task_a_rows": len(task_a),
        "task_b_rows": len(task_b),
        "duplicate_groups": int(reviews["duplicate_group_id"].nunique()),
        "empty_clean_texts": int(reviews["clean_text_empty"].sum()),
    },
    "outputs": {},
    "methodological_notes": [
        "Deterministic preprocessing only; no fitted corpus statistics.",
        "Negations, intensifiers, emoji semantics, and raw text are preserved.",
        "Duplicates are grouped and must remain in one split.",
        "Vectorizers must be fitted on training data only.",
        "Rating is not used as a sentiment target.",
    ],
}

for path in [review_output, task_a_output, task_b_output]:
    manifest["outputs"][str(path.relative_to(PROJECT_ROOT))] = {
        "sha256": sha256_file(path),
        "bytes": path.stat().st_size,
    }

manifest_output.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Preprocessing selesai. Output:")
for path in [review_output, task_a_output, task_b_output, manifest_output]:
    print(f"- {path.relative_to(PROJECT_ROOT)}")

## 13. Catatan untuk tahap berikutnya

Notebook ini berhenti sebelum data splitting dan feature extraction.

Pada notebook modeling berikutnya:

1. Gunakan duplicate_group_id sebagai group agar teks identik tidak bocor antar-split.
2. Seluruh pasangan dari review_id yang sama wajib berada di split yang sama.
3. Jangan menggunakan rating sebagai label sentimen.
4. Fit TF-IDF atau CountVectorizer hanya pada training set.
5. Bandingkan text_clean, text_negation, text_no_stopwords, dan text_stemmed secara empiris.
6. Gunakan n-gram 1–2 agar frasa seperti tidak enak tetap tertangkap.
7. Laporkan macro F1, micro F1, support, confusion matrix, dan error analysis.
8. Simpan split ID agar SVM, Random Forest, dan Multinomial Naive Bayes dibandingkan pada data yang sama.